In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pycontrails import Fleet

In [2]:
dff = pd.read_parquet("../data/filed_trajectories_EAGWP100.parquet")
fleetf = Fleet(data=dff)
print(f"Fleet contains {fleetf.n_flights} flights")

dfo = pd.read_parquet("../data/optimised_trajectories_EAGWP100.parquet")
fleeto = Fleet(data=dfo)
print(f"Fleet contains {fleeto.n_flights} flights")

Fleet contains 4112 flights
Fleet contains 4112 flights


In [3]:
# TODO fuel burn? nox? depends on if I want to have cruise only
# cols = ["fuel_burn", "nox", "NOx", "O3", "CH4", "H2O"]
cols = ["NOx", "O3", "CH4", "H2O"]  # for 3.1?
# cols = ["nox", "NOx", "O3", "CH4", "H2O"]  # for 3.2

from cane.utils import mask_by_marker

mask_by_marker(fleetf, cols)
mask_by_marker(fleeto, cols)

dff = fleetf.dataframe
dfo = fleeto.dataframe

from cane.utils import mask_by_validity_range

bounds = [150, 350]
mask_by_validity_range(fleetf, cols, bounds)
mask_by_validity_range(fleeto, cols, bounds)

dff = fleetf.dataframe
dfo = fleeto.dataframe

Marked rows filtered out for NOx
Marked rows filtered out for O3
Marked rows filtered out for CH4
Marked rows filtered out for H2O
Marked rows filtered out for NOx
Marked rows filtered out for O3
Marked rows filtered out for CH4
Marked rows filtered out for H2O
Bounds of [150, 350] in place for NOx
Bounds of [150, 350] in place for O3
Bounds of [150, 350] in place for CH4
Bounds of [150, 350] in place for H2O
Bounds of [150, 350] in place for NOx
Bounds of [150, 350] in place for O3
Bounds of [150, 350] in place for CH4
Bounds of [150, 350] in place for H2O


In [4]:
dff["CO2_CoCiP"] = dff["CO2"] + dff["CoCiP"]
dfo["CO2_CoCiP"] = dfo["CO2"] + dfo["CoCiP"]

dff["Total"] = dff["CO2"] + dff["CoCiP"] + dff["NOx"] + dff["H2O"]
dfo["Total"] = dfo["CO2"] + dfo["CoCiP"] + dfo["NOx"] + dfo["H2O"]

dff["NOx_H2O"] = dff["NOx"] + dff["H2O"]
dfo["NOx_H2O"] = dfo["NOx"] + dfo["H2O"]

In [5]:
from cane.utils import df_diff

# create aggregates (total)
org_sum = (
    dff[["flight_id", "fuel_burn", "nox", "co2", "NOx", "CoCiP", "CO2", "H2O", "Total", "CO2_CoCiP",
         "NOx_H2O"]]
    .groupby(["flight_id"]).sum().reset_index()
)
opt_sum = (
    dfo[["flight_id", "fuel_burn", "nox", "co2", "NOx", "CoCiP", "CO2", "H2O", "Total", "CO2_CoCiP",
         "NOx_H2O"]]
    .groupby(["flight_id"]).sum().reset_index()
)
my_diff = df_diff(org_sum, opt_sum, on=["flight_id"], keep_originals=True)
my_diff = pd.merge(
    my_diff,
    dff.groupby("flight_id").day.first(),
    on="flight_id",
)

In [6]:
# Sect 4
fids = my_diff.query("CO2_CoCiP_diff < 0 and Total_diff > 0").flight_id.unique()
len(fids)

43

In [7]:
meta = pd.read_parquet("../data/filed_meta.parquet")

print(meta.query("flight_id in @fids").day_pair.unique(), "days")
print(meta.query("flight_id in @fids").typecode.unique(), "aircraft types")

[1 3 4 5 6 7 8] days
<ArrowStringArray>
['B77W', 'A333', 'A319', 'A20N', 'B744', 'A339', 'A35K', 'B738', 'A320',
 'A388', 'GLEX', 'A343', 'B772', 'A346', 'A332', 'B77L', 'BCS3', 'A359']
Length: 18, dtype: str aircraft types
